# Whole-Ride Trajectory Maps

Maps the rover's position over the **entire ride**, on one figure:

* **original trajectory** &mdash; the real driven path. Raw FrodoBots GPS is
  jumpy (cheap single-frequency GNSS, ~1 Hz, urban multipath), so this notebook
  **fuses GPS position with an IMU/odometry heading** via a complementary filter
  (Section 7): high-frequency shape from odometry, slow absolute position from
  GPS. The cleaned raw GPS fixes are still drawn faintly underneath.
* **counterfactual: mild / strong** &mdash; the two perturbations from
  `02_counterfactual_generation.ipynb`, extended across the whole ride (Section 6).
* **MBRA dead-reckoned** &mdash; the MBRA-predicted `(v, ω)` integrated (notebook 2
  Section 4); what the counterfactuals perturb.

**Run this after notebooks 1 and 2, in the same Colab session.** It reads the
Drive-persisted `inputs/` (raw per-ride CSVs incl. `gps_data`) and `outputs/`
(`*_mbra_labels.csv`) those notebooks write, pulls each ride's `imu_data` from
the dataset zip (notebook 1 doesn't keep it), and saves one folder per ride
under `Drive/MBRA_reannotate/trajectory_maps/`.

## 1. Setup

Mount Drive and set the same paths notebooks 1 and 2 use. `utm` projects GPS
to metres (also a `deployment/` dependency); `requests` (preinstalled) pulls the
IMU logs via HTTP range requests, the same trick notebook 1 uses for the zip.

In [1]:
!pip install ipykernel
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Using cached huggingface_hub-1.29.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.29.0-py3-none-any.whl (795 kB)

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install -r Learning-to-Drive-Anywhere-with-MBRA/requirements.txt

import glob
import json
import math
import os
import struct
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/MBRA_reannotate"
INPUTS_DIR = os.path.join(DRIVE_ROOT, "inputs")          # raw per-ride CSVs (gps_data lives here) -- notebook 1
OUTPUTS_DIR = os.path.join(DRIVE_ROOT, "outputs")        # <ride_id>_mbra_labels.csv -- notebook 1
MAPS_DIR = os.path.join(DRIVE_ROOT, "trajectory_maps")   # this notebook's output root
os.makedirs(MAPS_DIR, exist_ok=True)

# Same zip notebook 1 sampled its rides from -- used here only to fetch imu_data.
DATASET_URL = "https://frodobots-2k-dataset.s3.ap-southeast-1.amazonaws.com/output_rides_0.zip"

# Counterfactual window params. TH_HISTORY / T_FUTURE match
# 02_counterfactual_generation.ipynb Section 1. `strong` is lowered from the
# notebook's 0.45 to 0.30 -- at 0.45 the per-window recovery routinely runs out
# of angular-velocity budget and leaves a residual that accumulates over the
# ride. On top of that, ENSURE_WINDOW_RECOVERY shrinks a window's alpha further
# whenever full recovery still wouldn't fit, so every window returns to the real
# trajectory and the whole-ride endpoint converges.
TH_HISTORY = 10
T_FUTURE = 20
WINDOW_N = TH_HISTORY + T_FUTURE
PERTURBATION_LEVELS = {"mild": 0.15, "strong": 0.30}
MAX_ADDED_RECOVERY_ANGULAR_VEL_RADPS = 0.5
ENSURE_WINDOW_RECOVERY = True      # per-window: shrink alpha until the recovery fully lands back on the real path
WINDOW_RECOVERY_TOL_M = 0.05

# --- GPS / IMU fusion (Section 7) ---
FUSE_GPS = True             # False -> plot cleaned raw GPS as the original trajectory, no fusion
USE_IMU_HEADING = True      # use imu_data compass/gyro as the heading source; else fall back to MBRA ω
FUSION_TAU_S = 20.0         # complementary-filter time constant (s): GPS wins slower than this, odometry faster
GPS_MAX_SPEED_MPS = 10.0    # gross-outlier gate: drop a fix only if it jumps faster than this away from BOTH neighbours
                            #   (catches teleport spikes; ordinary few-metre noise is left for the low-pass to handle)

# --- Scale sanity (Section 7/9) ---
# MBRA's forward() returns linear_vel already in m/s (max_linvel=0.5) and
# angular_vel in rad/s (max_angvel=1.0), with `normalize: False` -- see
# train/vint_train/models/exaug/exaug.py. So integrating v*dt with the real
# per-frame dt should give a metric path, and the GPS fit's scale should sit
# near 1.0. LOCK_SCALE forces a rigid (scale=1) fit so a unit/decoding bug
# upstream shows up as large RMSE instead of being silently absorbed.
LOCK_SCALE = False
SCALE_WARN_RANGE = (0.5, 2.0)      # warn if the fitted GPS scale (or MBRA/GPS path-length ratio) falls outside this

ride_ids_with_labels = sorted(
    os.path.basename(p)[: -len("_mbra_labels.csv")]
    for p in glob.glob(os.path.join(OUTPUTS_DIR, "*_mbra_labels.csv"))
)
print(f"{len(ride_ids_with_labels)} ride(s) with saved MBRA labels:", ride_ids_with_labels)


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'Learning-to-Drive-Anywhere-with-MBRA/requirements.txt'


ModuleNotFoundError: No module named 'google'

## 2. Load each ride's labels + GPS

`<ride_id>_mbra_labels.csv` is notebook 1's output (one row per labelled frame,
with `timestamp`, `mbra_linear_vel_mps`, `mbra_angular_vel_radps`). The raw
`gps_data_*.csv` sits next to `control_data` under `inputs/<ride_id>/`, also
pulled from the dataset zip by notebook 1. Column names in `gps_data` vary
between dataset dumps, so they're matched case-insensitively.

In [ ]:
def _match_col(raw, *cands):
    cols = {c.lower().strip(): c for c in raw.columns}
    for c in cands:
        if c in cols:
            return cols[c]
    return None


def _to_epoch_s(series):
    """Normalise a timestamp column to float seconds-since-epoch, whatever it
    arrives as: numeric s / ms / us / ns, or an ISO-8601 datetime string.
    Notebook 2's dead reckoning assumes the label timestamps are already in
    seconds; this brings gps_data / imu_data onto the same footing."""
    num = pd.to_numeric(series, errors="coerce")
    if num.notna().mean() < 0.5:                      # mostly non-numeric -> try datetime strings
        dt = pd.to_datetime(series, errors="coerce", utc=True)
        num = pd.Series(dt.view("int64"), index=series.index).where(dt.notna()) / 1e9
    med = num.dropna().abs().median()
    if med > 1e17:      num = num / 1e9               # nanoseconds
    elif med > 1e14:    num = num / 1e6               # microseconds
    elif med > 1e11:    num = num / 1e3               # milliseconds
    return num


def load_labels(ride_id):
    path = os.path.join(OUTPUTS_DIR, f"{ride_id}_mbra_labels.csv")
    assert os.path.exists(path), f"{path} not found -- run notebook 1 for {ride_id} first"
    return pd.read_csv(path).sort_values("frame_id").reset_index(drop=True)


def load_gps(ride_id):
    """Raw gps_data_*.csv -> timestamp (epoch s) / latitude / longitude, sorted."""
    matches = glob.glob(os.path.join(INPUTS_DIR, ride_id, "gps_data_*.csv"))
    assert matches, f"no gps_data_*.csv under {os.path.join(INPUTS_DIR, ride_id)}"
    raw = pd.read_csv(matches[0])
    ts = _match_col(raw, "timestamp", "time", "t", "unix_time", "utc", "datetime", "gps_time")
    lat = _match_col(raw, "latitude", "lat")
    lon = _match_col(raw, "longitude", "lon", "lng", "long")
    assert None not in (ts, lat, lon), f"gps_data columns not recognised: {list(raw.columns)}"
    out = pd.DataFrame({
        "timestamp": _to_epoch_s(raw[ts]),
        "latitude": pd.to_numeric(raw[lat], errors="coerce"),
        "longitude": pd.to_numeric(raw[lon], errors="coerce"),
    }).dropna()
    out = out[out["latitude"].between(-90, 90) & out["longitude"].between(-180, 180)]
    return out.sort_values("timestamp").reset_index(drop=True)

## 3. Fetch + load the IMU log

Notebook 1 only keeps `control_data` / `front_camera_timestamps` / `gps_data`
from the dataset zip, so `imu_data_*.csv` isn't on Drive. We pull just that one
member per ride straight out of the remote zip via an HTTP range request into
its compressed bytes (the same technique notebook 1 uses to read the zip
directory), and cache it under `inputs/<ride_id>/` so it downloads once.

FrodoBots `imu_data` is **batched JSON**: columns `compass`, `accelerometer`,
`gyroscope` (each cell a JSON array of `[v1, v2, v3, t_seconds]` readings, many
per row) plus a row `timestamp`. `load_imu` flattens each sensor stream; the
heading comes from the **compass** (3-axis magnetometer &rarr;
`atan2(m_y, m_x)`, drift-free) with the gyroscope as a fallback. Raw
magnetometer is uncalibrated (hard/soft-iron), so Section 7 only lets it
correct *slow* heading drift &mdash; the fast shape stays with MBRA's `ω`.

In [ ]:
def _fetch_range(url, start, end):
    r = requests.get(url, headers={"Range": f"bytes={start}-{end}"})
    r.raise_for_status()
    return r.content


def _zip64(extra, usize, csize, local_offset):
    off = 0
    while off + 4 <= len(extra):
        tag, size = struct.unpack("<HH", extra[off:off + 4])
        data = extra[off + 4:off + 4 + size]
        if tag == 0x0001:
            p = 0
            if usize == 0xFFFFFFFF:
                (usize,) = struct.unpack_from("<Q", data, p); p += 8
            if csize == 0xFFFFFFFF:
                (csize,) = struct.unpack_from("<Q", data, p); p += 8
            if local_offset == 0xFFFFFFFF:
                (local_offset,) = struct.unpack_from("<Q", data, p); p += 8
            break
        off += 4 + size
    return usize, csize, local_offset


def _list_zip_entries(url):
    total = int(requests.head(url).headers["Content-Length"])
    tail = _fetch_range(url, max(0, total - 65536), total - 1)
    i = tail.rfind(b"PK\x05\x06")
    (_, _, _, _, total_entries, cd_size, cd_offset, _) = struct.unpack("<IHHHHIIH", tail[i:i + 22])
    if cd_offset == 0xFFFFFFFF:
        j = tail.rfind(b"PK\x06\x07")
        (_, z64_off, _) = struct.unpack("<IQI", tail[j + 4:j + 20])
        z64 = _fetch_range(url, z64_off, z64_off + 55)
        (_, _, _, _, _, _, total_entries, cd_size, cd_offset) = struct.unpack("<QHHIIQQQQ", z64[4:56])
    cd = _fetch_range(url, cd_offset, cd_offset + cd_size - 1)
    entries, o = [], 0
    while o + 4 <= len(cd) and cd[o:o + 4] == b"PK\x01\x02":
        fixed = cd[o + 4:o + 46]
        (_, _, _, method, _, _, _, csize, usize, n_len, x_len, c_len, _, _, _, local_offset) = \
            struct.unpack("<HHHHHHIIIHHHHHII", fixed)
        name = cd[o + 46:o + 46 + n_len].decode("utf-8", "replace")
        extra = cd[o + 46 + n_len:o + 46 + n_len + x_len]
        if 0xFFFFFFFF in (usize, csize, local_offset):
            usize, csize, local_offset = _zip64(extra, usize, csize, local_offset)
        entries.append({"name": name, "method": method, "csize": csize, "usize": usize,
                        "local_offset": local_offset})
        o += 46 + n_len + x_len + c_len
    return entries


def _download_zip_entry(url, entry, pad=512):
    start = entry["local_offset"]
    buf = _fetch_range(url, start, start + 30 + pad + entry["csize"] - 1)
    (_, _, method, _, _, _, _, _, n_len, x_len) = struct.unpack("<HHHHHIIIHH", buf[4:30])
    ds = 30 + n_len + x_len
    raw = buf[ds:ds + entry["csize"]]
    if len(raw) < entry["csize"]:
        buf = _fetch_range(url, start, start + ds + entry["csize"] - 1)
        raw = buf[ds:ds + entry["csize"]]
    return zlib.decompress(raw, -15) if method == 8 else raw


_DATASET_ENTRIES = None


def ensure_imu_csv(ride_id):
    """Return a local path to the ride's imu_data_*.csv, downloading it from
    DATASET_URL on first use. None if the ride isn't in that zip / has no IMU."""
    global _DATASET_ENTRIES
    hit = glob.glob(os.path.join(INPUTS_DIR, ride_id, "imu_data_*.csv"))
    if hit:
        return hit[0]
    try:
        if _DATASET_ENTRIES is None:
            print("  reading dataset zip directory (once)...")
            _DATASET_ENTRIES = _list_zip_entries(DATASET_URL)
        cand = [e for e in _DATASET_ENTRIES
                if f"/{ride_id}/" in e["name"]
                and os.path.basename(e["name"]).startswith("imu_data")
                and e["name"].endswith(".csv")]
        if not cand:
            return None
        out_path = os.path.join(INPUTS_DIR, ride_id, os.path.basename(cand[0]["name"]))
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        with open(out_path, "wb") as f:
            f.write(_download_zip_entry(DATASET_URL, cand[0]))
        return out_path
    except Exception as e:
        print(f"  imu fetch failed for {ride_id}: {e}")
        return None


def _parse_sensor_col(series):
    """A batched-JSON IMU column -> (M,4) float array of [v1,v2,v3,t_seconds]
    readings, every row's readings concatenated, sorted + de-duped by timestamp."""
    rows = []
    for cell in series.dropna():
        try:
            arr = json.loads(cell) if isinstance(cell, str) else cell
        except (json.JSONDecodeError, TypeError):
            continue
        for e in arr or []:
            if isinstance(e, (list, tuple)) and len(e) >= 4:
                rows.append(e[:4])
    if not rows:
        return np.empty((0, 4))
    a = np.asarray(rows, dtype=np.float64)
    a = a[np.isfinite(a).all(axis=1)]
    a = a[np.argsort(a[:, 3])]
    _, keep = np.unique(a[:, 3], return_index=True)
    return a[keep]


def load_imu(path):
    """FrodoBots imu_data (batched-JSON compass/accelerometer/gyroscope columns)
    -> {'compass': df[timestamp,mx,my,mz], 'gyro': df[timestamp,gx,gy,gz]} or None."""
    if not path or not os.path.exists(path):
        return None
    raw = pd.read_csv(path)
    out = {}
    comp_col = _match_col(raw, "compass", "magnetometer", "mag")
    gyro_col = _match_col(raw, "gyroscope", "gyro", "angular_velocity")
    if comp_col is not None:
        m = _parse_sensor_col(raw[comp_col])
        if len(m) >= 2:
            out["compass"] = pd.DataFrame({"timestamp": m[:, 3], "mx": m[:, 0], "my": m[:, 1], "mz": m[:, 2]})
    if gyro_col is not None:
        g = _parse_sensor_col(raw[gyro_col])
        if len(g) >= 2:
            out["gyro"] = pd.DataFrame({"timestamp": g[:, 3], "gx": g[:, 0], "gy": g[:, 1], "gz": g[:, 2]})
    return out or None

## 4. GPS cleaning + projection to local metres

`clean_gps` drops "null island" `(0, 0)` fixes (a common GPS-error sentinel),
de-duplicates timestamps, and removes isolated teleport spikes (a fix that jumps
faster than `GPS_MAX_SPEED_MPS` away from **both** neighbours). Ordinary
few-metre jitter is left alone &mdash; the Section 7 low-pass handles it.
`gps_to_local_xy` then projects lat/lon to a local ENU plane in metres, origin
at the first fix &mdash; `utm` when available, else an equirectangular
approximation (both fine over one ride's ~100 m span).

In [ ]:
def gps_to_local_xy(lat, lon):
    lat = np.asarray(lat, dtype=np.float64)
    lon = np.asarray(lon, dtype=np.float64)
    try:
        import utm
        east, north, _, _ = utm.from_latlon(lat, lon)
        east = np.asarray(east, dtype=np.float64)
        north = np.asarray(north, dtype=np.float64)
    except Exception:  # utm missing, or a fix straddles a UTM zone boundary
        R = 6_378_137.0
        east = np.radians(lon - lon[0]) * math.cos(math.radians(lat[0])) * R
        north = np.radians(lat - lat[0]) * R
    return np.stack([east - east[0], north - north[0]], axis=1)


def clean_gps(gps_df, max_speed_mps=GPS_MAX_SPEED_MPS):
    g = gps_df[(gps_df["latitude"] != 0.0) | (gps_df["longitude"] != 0.0)]        # drop null-island fixes
    g = g.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)
    if len(g) < 3:
        return g
    xy = gps_to_local_xy(g["latitude"].to_numpy(), g["longitude"].to_numpy())
    ts = g["timestamp"].to_numpy()
    good = np.ones(len(g), dtype=bool)
    for i in range(1, len(g) - 1):
        dt_p, dt_n = ts[i] - ts[i - 1], ts[i + 1] - ts[i]
        far_prev = dt_p > 0 and math.hypot(*(xy[i] - xy[i - 1])) > max_speed_mps * dt_p
        far_next = dt_n > 0 and math.hypot(*(xy[i + 1] - xy[i])) > max_speed_mps * dt_n
        good[i] = not (far_prev and far_next)   # isolated teleport -> drop; consistent noise -> keep
    if good.sum() < 3:                          # never let the spike filter empty a non-empty log
        return g
    return g.iloc[good].reset_index(drop=True)


def gps_label_overlap(gps_df, label_ts):
    """(n_clean_fixes, gps_tspan, label_tspan, n_label_rows_inside_gps) -- for diagnostics."""
    g = clean_gps(gps_df)
    if len(g) == 0:
        return 0, (None, None), (float(label_ts.min()), float(label_ts.max())), 0
    gts = g["timestamp"].to_numpy()
    inside = int(((label_ts >= gts[0]) & (label_ts <= gts[-1])).sum())
    return len(g), (float(gts[0]), float(gts[-1])), (float(label_ts.min()), float(label_ts.max())), inside

## 5. Dead-reckon the MBRA trajectory

Verbatim from `02_counterfactual_generation.ipynb` Section 4: integrate the
MBRA-predicted `(v, ω)` frame-by-frame with each row's real timestamp delta.

**Units.** `notebook 1` stores `mbra_linear_vel_mps` / `mbra_angular_vel_radps`
= the **first step** of MBRA's 8-step action chunk (`linear_vel[0,0]`,
`angular_vel[0,0]`), one per video frame. `ExAug_dist_delay.forward`
(`train/vint_train/models/exaug/exaug.py`) returns these already in SI units --
`linear_vel = max_linvel * sigmoid(...)` with `max_linvel = 0.5` m/s,
`angular_vel = max_angvel * 2 * (sigmoid(...) - 0.5)` with `max_angvel = 1.0`
rad/s -- and `MBRA.yaml` has `normalize: False`. So there is **nothing to
un-normalise**, and integrating `v·dt` with the real per-frame `dt` yields a
metric path. If the GPS fit in Section 7 needs a scale far from 1 (or the
MBRA/GPS path-length ratio is far from 1), that's a red flag for the *frame
timestamps* (unit/rate) coming out of notebook 1, or for the ride genuinely
being near-stationary while MBRA still predicts motion toward the visual goal
-- not for this integration. `LOCK_SCALE = True` forces a rigid fit so such a
bug surfaces as large RMSE instead of being absorbed.

The reference rollout `robot_pos_model_fix` in `train_utils.py` lays out one
chunk over a fixed `dt = 0.333` s (3 Hz); here we instead re-sample MBRA's
*first* action every frame and integrate at the frame rate -- the MPC-style
"predict chunk, execute first action, re-plan" reconstruction, which is the
right quantity for a whole-ride path rather than one 2.7 s chunk.

In [ ]:
def integrate_ride_poses(labels_df):
    ts = labels_df["timestamp"].to_numpy()
    v = labels_df["mbra_linear_vel_mps"].to_numpy()
    w = labels_df["mbra_angular_vel_radps"].to_numpy()
    dt = np.diff(ts, prepend=ts[0])
    dt[0] = 0.0

    poses = np.zeros((len(labels_df), 3))
    x, y, theta = 0.0, 0.0, 0.0
    for i in range(len(labels_df)):
        theta += w[i] * dt[i]
        x += v[i] * math.cos(theta) * dt[i]
        y += v[i] * math.sin(theta) * dt[i]
        poses[i] = (x, y, theta)
    return poses

## 6. Whole-ride counterfactuals

Notebook 2 perturbs a single 30-row window (`TH_HISTORY=10` drift +
`T_FUTURE=20` recover). To cover a whole ride:

1. **Sample windows exactly as notebook 2 does** &mdash; one every `T_FUTURE`
   rows, so consecutive `WINDOW_N`-row windows overlap by `TH_HISTORY` rows.
2. Run the notebook's per-window logic on each: the `perturbation_heading_deviation`
   drift-then-recover profile, plus the `solve_recovery_correction` &gamma; term
   solved against that window's own poses/timestamps.
3. **Merge the overlapping windows** with a partition-of-unity trapezoidal
   weight (`_overlap_weight`): flat `= 1` over each window's central rows,
   ramping linearly across the `TH_HISTORY`-row overlaps so the two windows
   covering an overlap cross-fade to a sum of 1. Each window owns its central
   rows; the seams stay continuous. (Committing *only* the central rows without
   the cross-fade would step-jump the heading &mdash; those rows sit at the
   *peak* of the deviation, not at zero.)
4. Forward-integrate the merged heading-deviation track **once** with the real
   per-step MBRA velocity, so it collapses onto the real trajectory on the ride
   tail past the last window.

Only heading is perturbed; linear velocity always matches the real trajectory.

In [ ]:
def perturbation_heading_deviation(alpha_rad, n_total, n_history):  # notebook 2, Section 6
    n_future = n_total - n_history
    dev = np.empty(n_total)
    hist_idx = np.arange(n_history)
    dev[:n_history] = alpha_rad * 0.5 * (1 - np.cos(np.pi * hist_idx / (n_history - 1)))
    fut_idx = np.arange(n_history, n_total) - (n_history - 1)
    dev[n_history:] = alpha_rad * 0.5 * (1 + np.cos(np.pi * fut_idx / n_future))
    return dev


def recovery_correction_shape(n_total, n_history):  # notebook 2, Section 7
    n_future = n_total - n_history
    shape = np.zeros(n_total)
    u = np.arange(n_future + 1)
    shape[n_history - 1:] = np.sin(np.pi * u / n_future)
    return shape


def apply_lateral_offset(real_poses, real_v, heading_deviation, dts):  # notebook 2, Section 6
    n = len(real_poses)
    theta_new = real_poses[:, 2] + heading_deviation
    x, y = np.zeros(n), np.zeros(n)
    x[0], y[0] = real_poses[0, 0], real_poses[0, 1]
    for i in range(1, n):
        x[i] = x[i - 1] + real_v[i - 1] * math.cos(theta_new[i - 1]) * dts[i - 1]
        y[i] = y[i - 1] + real_v[i - 1] * math.sin(theta_new[i - 1]) * dts[i - 1]
    return np.stack([x, y, theta_new], axis=1)


def poses_to_vw(poses, dts):  # notebook 2, Section 7
    vw = []
    for (x0, y0, th0), (x1, y1, th1), dt in zip(poses[:-1], poses[1:], dts):
        v = math.hypot(x1 - x0, y1 - y0) / dt
        dtheta = math.atan2(math.sin(th1 - th0), math.cos(th1 - th0))
        vw.append((v, dtheta / dt))
    return vw


def solve_recovery_correction(real_poses, real_v, base_deviation, dts, correction_shape,
                              n_history, max_added_w=MAX_ADDED_RECOVERY_ANGULAR_VEL_RADPS):  # notebook 2, Section 7
    def poses_for(gamma):
        return apply_lateral_offset(real_poses, real_v, base_deviation + gamma * correction_shape, dts)

    def lateral_error(gamma):
        poses = poses_for(gamma)
        dx = poses[-1, 0] - real_poses[-1, 0]
        dy = poses[-1, 1] - real_poses[-1, 1]
        theta_f = real_poses[-1, 2]
        return -math.sin(theta_f) * dx + math.cos(theta_f) * dy

    def max_abs_w(gamma):
        vw = poses_to_vw(poses_for(gamma)[n_history - 1:], dts[n_history - 1:])
        return max(abs(w) for _, w in vw)

    g0, g1 = 0.0, 0.01
    f0, f1 = lateral_error(g0), lateral_error(g1)
    for _ in range(25):
        if abs(f1) < 1e-9 or f1 == f0:
            break
        g2 = g1 - f1 * (g1 - g0) / (f1 - f0)
        g0, f0 = g1, f1
        g1, f1 = g2, lateral_error(g2)
    gamma_full = g1

    budget = max_abs_w(0.0) + max_added_w
    if max_abs_w(gamma_full) <= budget:
        return gamma_full
    lo, hi = 0.0, gamma_full
    for _ in range(40):
        mid = (lo + hi) / 2
        if max_abs_w(mid) <= budget:
            lo = mid
        else:
            hi = mid
    return lo


def _overlap_weight(window_n, overlap):
    """Trapezoidal partition-of-unity weight: ramp up over the first `overlap`
    rows, flat 1 in the middle, ramp down over the last `overlap`. Two windows
    staggered by `window_n - overlap` have complementary ramps summing to 1."""
    w = np.ones(window_n)
    ramp = (np.arange(overlap) + 0.5) / overlap
    w[:overlap] = ramp
    w[-overlap:] = ramp[::-1]
    return w


def _window_end_lateral(wp, wv, dev, wdt):
    """Perpendicular offset of the perturbed window's last pose from the real
    window's last pose -- 0 means the window recovered onto the real path."""
    p = apply_lateral_offset(wp, wv, dev, wdt)
    dx, dy = p[-1, 0] - wp[-1, 0], p[-1, 1] - wp[-1, 1]
    thf = wp[-1, 2]
    return -math.sin(thf) * dx + math.cos(thf) * dy


def build_counterfactual_track(labels_df, real_poses, alpha_rad,
                               window_n=WINDOW_N, th_history=TH_HISTORY,
                               ensure_recovery=ENSURE_WINDOW_RECOVERY,
                               recover_tol_m=WINDOW_RECOVERY_TOL_M):
    ts = labels_df["timestamp"].to_numpy()
    v = labels_df["mbra_linear_vel_mps"].to_numpy()
    n = len(labels_df)
    dt = np.diff(ts, prepend=ts[0])
    dt[0] = 0.0

    base_full = perturbation_heading_deviation(alpha_rad, window_n, th_history)
    corr = recovery_correction_shape(window_n, th_history)
    weight = _overlap_weight(window_n, th_history)
    stride = window_n - th_history  # notebook 2's window sampling stride (== T_FUTURE)

    dev_acc = np.zeros(n)
    wt_acc = np.zeros(n)
    gammas, eff_alpha = [], []
    for start in range(0, n - window_n + 1, stride):
        sl = slice(start, start + window_n)
        wp, wv, wdt = real_poses[sl], v[sl], np.diff(ts[sl])
        k = 1.0
        for _ in range(16):
            base = k * base_full
            gamma = solve_recovery_correction(wp, wv, base, wdt, corr, th_history)
            if (not ensure_recovery
                    or abs(_window_end_lateral(wp, wv, base + gamma * corr, wdt)) <= recover_tol_m):
                break
            k *= 0.8            # recovery ran out of ω budget -> weaken this window's perturbation
        dev_acc[sl] += weight * (base + gamma * corr)
        wt_acc[sl] += weight
        gammas.append(float(gamma))
        eff_alpha.append(float(k * alpha_rad))

    global_dev = np.zeros(n)
    covered = wt_acc > 0
    global_dev[covered] = dev_acc[covered] / wt_acc[covered]

    theta_new = real_poses[:, 2] + global_dev
    x = np.zeros(n)
    y = np.zeros(n)
    for i in range(1, n):
        x[i] = x[i - 1] + v[i] * math.cos(theta_new[i]) * dt[i]
        y[i] = y[i - 1] + v[i] * math.sin(theta_new[i]) * dt[i]

    return {"poses": np.stack([x, y, theta_new], axis=1),
            "heading_deviation": global_dev, "gammas": gammas,
            "effective_alpha_rad": eff_alpha}

## 7. Fuse GPS position with the IMU/odometry heading

A spatial-domain **complementary filter**:

1. **Odometry** &mdash; dead-reckon a smooth path from MBRA speed
   (`mbra_linear_vel_mps`) and a heading track: the IMU orientation yaw if
   available, else integrated IMU gyro-z, else MBRA's own `ω`
   (`USE_IMU_HEADING` picks whether to try the IMU at all).
2. **Rigid fit** &mdash; align the odometry to the cleaned GPS with a similarity
   transform (Umeyama: scale + rotation + translation) over the overlapping
   timestamps. Scale is fitted because MBRA speed isn't guaranteed
   metric-calibrated.
3. **Slow correction** &mdash; low-pass the residual `GPS − aligned-odometry`
   (zero-phase, time constant `FUSION_TAU_S`) and add it back. The result keeps
   odometry's high-frequency shape and GPS's low-frequency absolute position,
   and shrugs off individual GPS spikes.

The similarity transform from step 2 ties the two frames together: its inverse
brings the GPS / fused track into the **MBRA ego frame** (x forward, y left),
which is where everything is plotted (Section 9); the forward direction still
fills the `map_*` (GPS ENU) columns of the CSV.

In [ ]:
def umeyama_2d(src, dst, lock_scale=False):
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    mu_s, mu_d = src.mean(axis=0), dst.mean(axis=0)
    s0, d0 = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd((d0.T @ s0) / len(src))
    S = np.eye(2)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[1, 1] = -1.0
    R = U @ S @ Vt
    var_s = (s0 ** 2).sum() / len(src)
    scale = 1.0 if lock_scale else (float((D * np.diag(S)).sum() / var_s) if var_s > 0 else 1.0)
    if not np.isfinite(scale):
        scale = 0.0
    return scale, R, mu_d - scale * (R @ mu_s)


SCALE_DEGENERATE = (1e-3, 1e3)   # a fit outside this is unusable (MBRA path has ~no extent vs GPS)


def scale_is_usable(scale):
    return np.isfinite(scale) and SCALE_DEGENERATE[0] < abs(scale) < SCALE_DEGENERATE[1]


def apply_similarity(xy, scale, R, t):
    return scale * (np.asarray(xy, dtype=np.float64) @ R.T) + t


def invert_similarity(xy, scale, R, t):
    """Inverse of apply_similarity: map a point back from the fitted frame to the
    source (MBRA ego) frame. R is orthonormal so (A @ R.T) @ R == A."""
    xy = np.asarray(xy, dtype=np.float64)
    if not scale_is_usable(scale):
        return np.full_like(xy, np.nan)
    return ((xy - np.asarray(t)) @ R) / scale


def path_length(xy):
    xy = np.asarray(xy, dtype=np.float64)
    xy = xy[np.isfinite(xy).all(axis=1)]
    return float(np.hypot(*np.diff(xy, axis=0).T).sum()) if len(xy) > 1 else 0.0


def _ema(sig, dts, tau):
    out = np.array(sig, dtype=np.float64)
    for i in range(1, len(out)):
        a = 1.0 - math.exp(-dts[i] / tau) if dts[i] > 0 else 0.0
        out[i] = out[i - 1] + a * (sig[i] - out[i - 1])
    return out


def lowpass_zero_phase(sig, ts, tau):
    """Forward-backward 1st-order low pass over irregular timestamps (per column)."""
    sig = np.asarray(sig, dtype=np.float64)
    dts_f = np.diff(ts, prepend=ts[0]); dts_f[0] = 0.0
    dts_b = np.abs(np.diff(ts[::-1], prepend=ts[::-1][0]))
    out = np.empty_like(sig)
    for k in range(sig.shape[1]):
        f = _ema(sig[:, k], dts_f, tau)
        out[:, k] = _ema(f[::-1], dts_b, tau)[::-1]
    return out


def heading_track(label_ts, imu, fallback_theta, use_imu=True, tau_heading=15.0):
    """Heading (rad) at each label timestamp + a label for the source used.

    `fallback_theta` is MBRA's own dead-reckoned heading (smooth, high-rate, but
    drifts). With a compass we keep that shape and let the drift-free (but
    uncalibrated, low-rate) magnetometer yaw pull out the slow drift via a
    zero-phase low pass of the residual.
    """
    fb = np.asarray(fallback_theta, dtype=np.float64)
    fb = fb - fb[0]
    if not (use_imu and imu):
        return fb, "mbra-omega"

    if "compass" in imu:
        c = imu["compass"]
        mag_yaw = np.unwrap(np.arctan2(c["my"].to_numpy(), c["mx"].to_numpy()))
        theta_mag = np.interp(label_ts, c["timestamp"].to_numpy(), mag_yaw)
        d_mag, d_fb = np.diff(theta_mag), np.diff(fb)
        if np.dot(d_mag, d_fb) < 0:                       # mirrored axis convention -> flip
            theta_mag = -theta_mag
        resid = np.unwrap(theta_mag - fb)
        resid -= resid[0]
        theta = fb + lowpass_zero_phase(resid.reshape(-1, 1), label_ts, tau_heading).ravel()
        return theta, "compass+mbra-omega"

    if "gyro" in imu:
        g = imu["gyro"]
        gts = g["timestamp"].to_numpy()
        gz = g["gz"].to_numpy() - np.median(g["gz"].to_numpy())      # crude bias removal
        if np.percentile(np.abs(gz), 95) > 6.5:                       # values that large must be deg/s
            gz = np.radians(gz)
        yaw = np.concatenate([[0.0], np.cumsum(0.5 * (gz[1:] + gz[:-1]) * np.diff(gts))])
        th = np.interp(label_ts, gts, yaw)
        return th - th[0], "imu-gyro"

    return fb, "mbra-omega"


def fuse_gps_imu(labels_df, gps_df, imu_df, real_poses, tau=FUSION_TAU_S, use_imu=USE_IMU_HEADING):
    """-> (fused_xy (N,2) in GPS ENU, similarity transform (scale,R,t), info dict).
    fused_xy / transform are None if there aren't enough GPS fixes to anchor."""
    ts = labels_df["timestamp"].to_numpy()
    v = labels_df["mbra_linear_vel_mps"].to_numpy()
    dt = np.diff(ts, prepend=ts[0]); dt[0] = 0.0

    theta, src = heading_track(ts, imu_df, real_poses[:, 2], use_imu=use_imu)
    odo = np.stack([np.cumsum(v * np.cos(theta) * dt),
                    np.cumsum(v * np.sin(theta) * dt)], axis=1)

    g = clean_gps(gps_df)
    info = {"heading_source": src, "n_gps_kept": int(len(g)),
            "n_gps_rejected": int(len(gps_df) - len(g)), "tau_s": tau}
    if len(g) < 3:
        info["fused"] = False
        return None, None, info

    gxy = gps_to_local_xy(g["latitude"].to_numpy(), g["longitude"].to_numpy())
    gts = g["timestamp"].to_numpy()
    gps_on = np.stack([np.interp(ts, gts, gxy[:, 0]), np.interp(ts, gts, gxy[:, 1])], axis=1)
    inside = (ts >= gts[0]) & (ts <= gts[-1])
    if inside.sum() < 3:
        info["fused"] = False
        return None, None, info

    scale, R, t = umeyama_2d(odo[inside], gps_on[inside], lock_scale=LOCK_SCALE)
    if not scale_is_usable(scale):
        info.update(fused=False, scale=float(scale), degenerate_scale=True,
                    raw_mbra_path_len_m=path_length(odo[inside]),
                    gps_path_len_m=path_length(gps_on[inside]),
                    median_frame_dt_s=float(np.median(np.diff(ts))) if len(ts) > 1 else None)
        return None, None, info

    odo_a = apply_similarity(odo, scale, R, t)
    correction = lowpass_zero_phase(gps_on - odo_a, ts, tau)
    fused = odo_a + correction

    resid = fused[inside] - gps_on[inside]
    # `scale` is the key metric: MBRA v is already m/s, so odometry->GPS scale
    # should be ~1. raw_mbra_path_len (unscaled dead reckoning) vs fused_path_len
    # (~ the distance actually travelled) makes the same point in metres; their
    # ratio is ~1/scale. Raw-GPS cumulative length is noise-dominated, not shown.
    info.update(fused=True, scale=scale, lock_scale=bool(LOCK_SCALE),
                rotation_deg=math.degrees(math.atan2(R[1, 0], R[0, 0])),
                rmse_vs_gps_m=float(np.sqrt((resid ** 2).sum(axis=1).mean())),
                raw_mbra_path_len_m=path_length(odo[inside]),
                fused_path_len_m=path_length(fused[inside]),
                median_frame_dt_s=float(np.median(np.diff(ts))) if len(ts) > 1 else None)
    return fused, (scale, R, t), info

## 8. Fallback alignment (no fusion)

Used when `FUSE_GPS` is off or a ride has too few GPS fixes: a single similarity
fit of the MBRA dead-reckoned track onto the cleaned GPS, so the tracks still
share a frame.

In [ ]:
def align_track_to_gps(ego_xy, label_ts, gps_df):
    """-> (transform (scale,R,t) or None, fit-stats dict)."""
    g = clean_gps(gps_df)
    if len(g) < 3:
        return None, {"ok": False}
    gxy = gps_to_local_xy(g["latitude"].to_numpy(), g["longitude"].to_numpy())
    gts = g["timestamp"].to_numpy()
    on_labels = np.stack([np.interp(label_ts, gts, gxy[:, 0]),
                          np.interp(label_ts, gts, gxy[:, 1])], axis=1)
    ok = (label_ts >= gts[0]) & (label_ts <= gts[-1])
    if ok.sum() < 3:
        return None, {"ok": False}
    scale, R, t = umeyama_2d(ego_xy[ok], on_labels[ok], lock_scale=LOCK_SCALE)
    if not scale_is_usable(scale):
        return None, {"ok": False, "scale": float(scale), "degenerate_scale": True,
                      "raw_mbra_path_len_m": path_length(ego_xy[ok]),
                      "gps_path_len_m": path_length(on_labels[ok]),
                      "median_frame_dt_s": float(np.median(np.diff(label_ts))) if len(label_ts) > 1 else None}
    resid = apply_similarity(ego_xy[ok], scale, R, t) - on_labels[ok]
    return (scale, R, t), {
        "ok": True, "scale": scale, "lock_scale": bool(LOCK_SCALE),
        "rotation_deg": math.degrees(math.atan2(R[1, 0], R[0, 0])),
        "rmse_m": float(np.sqrt((resid ** 2).sum(axis=1).mean())), "n_samples": int(ok.sum()),
        "raw_mbra_path_len_m": path_length(ego_xy[ok]),
        "median_frame_dt_s": float(np.median(np.diff(label_ts))) if len(label_ts) > 1 else None,
    }

## 9. Build tracks, plot, and save &mdash; one folder per ride

For every ride with saved labels: fetch IMU, build the tracks, fuse, plot, and
write to `Drive/MBRA_reannotate/trajectory_maps/<ride_id>/`:

* `trajectories.csv` &mdash; tidy long table `track, frame_id, timestamp,
  ego_x, ego_y, ego_theta, map_x, map_y, heading_deviation_rad`.
  `track` &isin; {`original`, `gps_raw`, `mbra_real`, `cf_mild`, `cf_strong`}.
  **`ego_*` is the bird's-eye MBRA robot frame (x forward, y left) &mdash; every
  track, GPS remapped into it.** `map_*` is the same data in the GPS ENU frame.
* `trajectory_meta.json` &mdash; fusion / alignment / scale-sanity stats +
  per-counterfactual summary (incl. `max/mean_abs_lateral_offset_m`).
* `trajectory_map.png` &mdash; two panels:
  * **left** &mdash; whole-ride bird's-eye (MBRA ego frame): raw GPS (thin
    dotted line), fused original, MBRA dead-reckoned, both counterfactuals.
  * **right** &mdash; **counterfactual lateral offset from MBRA vs distance**.
    The drift-then-recover perturbations are sub-metre and vanish at whole-ride
    zoom, so this panel plots the signed perpendicular offset of each
    counterfactual from the MBRA path (MBRA = the flat zero line), auto-scaled
    so the mild-vs-strong difference and the per-window sawtooth are obvious.

In [ ]:
TRACK_STYLE = {
    "gps_raw":   dict(color="#9aa0a6", lw=0.9, ls="-",  alpha=0.55, zorder=1.0, dots=True,
                      label="raw GPS (cleaned)"),
    "original":  dict(color="#111111", lw=2.4, ls="-",  alpha=0.85, zorder=3.0,
                      label="original trajectory (GPS+IMU fused)"),
    "cf_mild":   dict(color="#2ca02c", lw=1.6, ls="--", alpha=0.95, zorder=2.0,
                      label=f"counterfactual: mild (α={PERTURBATION_LEVELS['mild']:g})"),
    "cf_strong": dict(color="#d62728", lw=1.6, ls="--", alpha=0.95, zorder=2.5,
                      label=f"counterfactual: strong (α={PERTURBATION_LEVELS['strong']:g})"),
    "mbra_real": dict(color="#1f77b4", lw=2.0, ls="-",  alpha=1.0,  zorder=4.0,
                      label="MBRA dead-reckoned"),
}
MAP_DRAW_ORDER = ("gps_raw", "original", "cf_mild", "cf_strong", "mbra_real")
LEGEND_RANK = {TRACK_STYLE[k]["label"]: i for i, k in enumerate(
    ("original", "mbra_real", "cf_mild", "cf_strong", "gps_raw"))}


def _draw(ax, xy, style, ends=True):
    xy = np.asarray(xy, dtype=np.float64)
    xy = xy[np.isfinite(xy).all(axis=1)]
    if len(xy) == 0:
        return
    ax.plot(xy[:, 0], xy[:, 1], color=style["color"], lw=style["lw"], ls=style["ls"],
            alpha=style["alpha"], zorder=style["zorder"], label=style["label"],
            marker=("." if style.get("dots") else None), ms=3, mew=0)
    if ends:
        ax.scatter(*xy[0], color=style["color"], marker="o", s=26, zorder=style["zorder"] + 0.1)
        ax.scatter(*xy[-1], color=style["color"], marker="s", s=26, zorder=style["zorder"] + 0.1)


def _legend(ax, **kw):
    h, l = ax.get_legend_handles_labels()
    hl = sorted(zip(h, l), key=lambda p: LEGEND_RANK.get(p[1], 9))
    ax.legend([x[0] for x in hl], [x[1] for x in hl], fontsize=8, framealpha=0.92, **kw)


def _cross_track(xy, ref_xy, ref_theta):
    """Signed perpendicular offset of `xy` from the reference path (+ = left of
    the reference heading). Rows must be frame-aligned with the reference."""
    d = np.asarray(xy, float) - np.asarray(ref_xy, float)
    return -np.sin(ref_theta) * d[:, 0] + np.cos(ref_theta) * d[:, 1]


def _cumdist(xy):
    xy = np.asarray(xy, float)
    return np.concatenate([[0.0], np.cumsum(np.hypot(*np.diff(xy, axis=0).T))])


def process_ride(ride_id):
    labels = load_labels(ride_id)
    gps = load_gps(ride_id)
    label_ts = labels["timestamp"].to_numpy()
    frame_id = labels["frame_id"].to_numpy()

    imu = load_imu(ensure_imu_csv(ride_id)) if (FUSE_GPS and USE_IMU_HEADING) else None
    real_poses = integrate_ride_poses(labels)
    cfs = {lvl: build_counterfactual_track(labels, real_poses, a)
           for lvl, a in PERTURBATION_LEVELS.items()}

    meta = {"ride_id": ride_id, "n_label_rows": int(len(labels))}
    fused, transform, finfo = (None, None, {"fused": False})
    if FUSE_GPS:
        fused, transform, finfo = fuse_gps_imu(labels, gps, imu, real_poses)
    meta["fusion"] = finfo

    if fused is not None:
        original_enu = fused                              # already GPS ENU
        original_label = f"original trajectory (GPS+IMU fused · heading: {finfo['heading_source']})"
        frame_note = (f"fused: heading {finfo['heading_source']}, τ={finfo['tau_s']:.0f}s, "
                      f"scale={finfo['scale']:.3f}, "
                      f"MBRA len {finfo['raw_mbra_path_len_m']:.0f} m vs actual {finfo['fused_path_len_m']:.0f} m, "
                      f"RMSE vs GPS {finfo['rmse_vs_gps_m']:.1f} m "
                      f"({finfo['n_gps_kept']} fixes, {finfo['n_gps_rejected']} rejected)")
    else:
        transform, fit = align_track_to_gps(real_poses[:, :2], label_ts, gps)
        meta["alignment"] = fit
        g = clean_gps(gps)
        original_enu = gps_to_local_xy(g["latitude"].to_numpy(), g["longitude"].to_numpy()) if len(g) else np.zeros((0, 2))
        original_label = "original trajectory (cleaned GPS)"
        if fit.get("ok"):
            frame_note = f"not fused — MBRA fit to cleaned GPS: scale={fit['scale']:.3f}, RMSE={fit['rmse_m']:.1f} m"
        elif finfo.get("degenerate_scale") or fit.get("degenerate_scale"):
            di = finfo if finfo.get("degenerate_scale") else fit
            ml, gl = di.get("raw_mbra_path_len_m"), di.get("gps_path_len_m")
            mdt = di.get("median_frame_dt_s")
            frame_note = (f"degenerate GPS fit (scale={di.get('scale', 0):.3g}) — "
                          f"MBRA path {ml:.1f} m vs GPS {gl:.1f} m; showing MBRA + counterfactuals only")
            meta["gps_diagnostic"] = {"reason": "degenerate_scale", **di}
            print(f"  {ride_id}: alignment scale {di.get('scale', 0):.3g} is degenerate — "
                  f"the MBRA dead-reckoned path spans only {ml:.1f} m while GPS spans {gl:.1f} m "
                  f"(median frame dt {mdt:.3g} s).")
            print(f"    -> either this ride is near-stationary, or v/dt from notebook 1 is mis-scaled "
                  f"(check mbra_linear_vel_mps and timestamp in {ride_id}_mbra_labels.csv). "
                  f"Not aligning; the deviation panel (MBRA vs counterfactuals) is still valid.")
        else:
            nfix, gspan, lspan, inside = gps_label_overlap(gps, label_ts)
            frame_note = "no usable GPS — MBRA & counterfactuals only"
            meta["gps_diagnostic"] = {"raw_rows": int(len(gps)), "clean_fixes": nfix,
                                      "gps_tspan": gspan, "label_tspan": lspan, "label_rows_inside_gps": inside}
            print(f"  {ride_id}: GPS unusable — {len(gps)} raw rows, {nfix} after cleaning, "
                  f"{inside} label rows inside GPS time span")
            print(f"    gps t: {gspan}   labels t: {lspan}   "
                  f"({'NO TIME OVERLAP — timestamp unit/epoch mismatch?' if inside == 0 and nfix else 'too few fixes'})")
    TRACK_STYLE["original"]["label"] = original_label

    # Everything is drawn in the MBRA ego frame (x forward, y left). The GPS /
    # fused track is remapped *into* that frame with the inverse of the fit;
    # MBRA + counterfactuals are already in it.
    to_ego = ((lambda enu: invert_similarity(enu, *transform)) if transform is not None
              else (lambda enu: np.full((len(np.atleast_2d(enu)), 2), np.nan)))
    to_map = ((lambda xy: apply_similarity(xy, *transform)) if transform is not None
              else (lambda xy: np.full_like(np.asarray(xy, float), np.nan)))

    # --- scale sanity: MBRA velocities are already m/s, so the GPS fit's scale
    #     and the MBRA/GPS path-length ratio should both sit near 1.0 ---
    sinfo = finfo if fused is not None else meta.get("alignment", {})
    lo, hi = SCALE_WARN_RANGE
    sc = sinfo.get("scale")
    mdt = sinfo.get("median_frame_dt_s")
    if sc is not None and not sinfo.get("degenerate_scale"):   # degenerate already reported above
        bits = [f"GPS-fit scale={sc:.3f}"]
        if sinfo.get("raw_mbra_path_len_m") is not None and sinfo.get("fused_path_len_m") is not None:
            bits.append(f"MBRA path {sinfo['raw_mbra_path_len_m']:.0f} m vs actual {sinfo['fused_path_len_m']:.0f} m")
        if mdt is not None:
            bits.append(f"median frame dt={mdt:.3f} s")
        print(f"  {ride_id}: " + ", ".join(bits))
        flags = []
        if not (lo <= sc <= hi):
            inv = f"1/scale = {1 / sc:.3g}x" if abs(sc) > 1e-9 else "scale ≈ 0"
            flags.append(f"scale {sc:.3g} outside [{lo}, {hi}] ({inv})")
        if mdt is not None and not (0.02 <= mdt <= 1.0):
            flags.append(f"frame dt {mdt:.3g} s (expected ~0.05 s at 20 Hz — check notebook 1 timestamp units)")
        if flags:
            print(f"    ⚠ scale sanity: {'; '.join(flags)}. "
                  f"MBRA (v,ω) are already SI (exaug.py max_linvel=0.5, normalize:False), so scale should be ~1; "
                  f"a big mismatch points at frame-timestamp units/rate or a near-stationary ride. "
                  f"Set LOCK_SCALE=True to stop the fit absorbing it.")
        meta["scale_sanity"] = {"gps_fit_scale": sc, "median_frame_dt_s": mdt, "flagged": bool(flags)}

    g_clean = clean_gps(gps)
    gps_raw_enu = (gps_to_local_xy(g_clean["latitude"].to_numpy(), g_clean["longitude"].to_numpy())
                   if len(g_clean) else np.zeros((0, 2)))
    gps_raw_ego = to_ego(gps_raw_enu) if len(gps_raw_enu) else np.zeros((0, 2))
    original_ego = to_ego(original_enu) if len(original_enu) else np.zeros((0, 2))

    parts = [
        pd.DataFrame({"track": "gps_raw", "frame_id": np.nan,
                      "timestamp": g_clean["timestamp"].to_numpy() if len(g_clean) else [],
                      "ego_x": gps_raw_ego[:, 0], "ego_y": gps_raw_ego[:, 1], "ego_theta": np.nan,
                      "map_x": gps_raw_enu[:, 0], "map_y": gps_raw_enu[:, 1], "heading_deviation_rad": np.nan}),
        pd.DataFrame({"track": "original", "frame_id": np.nan,
                      "timestamp": (label_ts if fused is not None
                                    else (g_clean["timestamp"].to_numpy() if len(g_clean) else [])),
                      "ego_x": original_ego[:, 0], "ego_y": original_ego[:, 1], "ego_theta": np.nan,
                      "map_x": original_enu[:, 0], "map_y": original_enu[:, 1], "heading_deviation_rad": np.nan}),
        pd.DataFrame({"track": "mbra_real", "frame_id": frame_id, "timestamp": label_ts,
                      "ego_x": real_poses[:, 0], "ego_y": real_poses[:, 1], "ego_theta": real_poses[:, 2],
                      "map_x": to_map(real_poses[:, :2])[:, 0], "map_y": to_map(real_poses[:, :2])[:, 1],
                      "heading_deviation_rad": 0.0}),
    ]
    cf_cross = {}
    for lvl, cf in cfs.items():
        cf_map = to_map(cf["poses"][:, :2])
        parts.append(pd.DataFrame({
            "track": f"cf_{lvl}", "frame_id": frame_id, "timestamp": label_ts,
            "ego_x": cf["poses"][:, 0], "ego_y": cf["poses"][:, 1], "ego_theta": cf["poses"][:, 2],
            "map_x": cf_map[:, 0], "map_y": cf_map[:, 1], "heading_deviation_rad": cf["heading_deviation"],
        }))
        ct = _cross_track(cf["poses"][:, :2], real_poses[:, :2], real_poses[:, 2])
        cf_cross[lvl] = ct
        ea = cf.get("effective_alpha_rad") or [PERTURBATION_LEVELS[lvl]]
        meta[f"cf_{lvl}"] = {
            "alpha_rad": PERTURBATION_LEVELS[lvl], "n_windows": len(cf["gammas"]),
            "effective_alpha_min": float(np.min(ea)), "effective_alpha_mean": float(np.mean(ea)),
            "n_windows_alpha_reduced": int(np.sum(np.asarray(ea) < PERTURBATION_LEVELS[lvl] - 1e-9)),
            "gamma_max_abs": float(np.max(np.abs(cf["gammas"]))) if cf["gammas"] else 0.0,
            "endpoint_offset_from_real_m": float(np.hypot(*(cf["poses"][-1, :2] - real_poses[-1, :2]))),
            "max_abs_lateral_offset_m": float(np.max(np.abs(ct))),
            "mean_abs_lateral_offset_m": float(np.mean(np.abs(ct))),
        }
    df = pd.concat(parts, ignore_index=True)

    # --- Figure: left = bird's-eye (whole ride), right = counterfactual lateral
    #     offset from MBRA vs distance -- the drift-then-recover perturbations are
    #     sub-metre, invisible at whole-ride zoom, so they get their own panel. ---
    fig, (ax_be, ax_dev) = plt.subplots(1, 2, figsize=(16, 7.5),
                                        gridspec_kw={"width_ratios": [1.15, 1]})

    for tk in MAP_DRAW_ORDER:
        _draw(ax_be, df.loc[df.track == tk, ["ego_x", "ego_y"]].to_numpy(), TRACK_STYLE[tk])
    ax_be.set_title(f"{ride_id} — bird's-eye (MBRA ego frame)\n{frame_note}", fontsize=9)
    ax_be.set_xlabel("x (m, forward)"); ax_be.set_ylabel("y (m, left)")
    ax_be.set_aspect("equal", adjustable="datalim")
    ax_be.grid(alpha=0.25)
    _legend(ax_be, loc="upper left")

    s = _cumdist(real_poses[:, :2])
    ax_dev.axhline(0, color=TRACK_STYLE["mbra_real"]["color"], lw=1.8, label="MBRA (reference)")
    for lvl in PERTURBATION_LEVELS:
        st = TRACK_STYLE[f"cf_{lvl}"]
        ax_dev.plot(s, cf_cross[lvl], color=st["color"], lw=1.6, ls=st["ls"], label=st["label"])
    def _dev_line(lvl):
        m = meta[f"cf_{lvl}"]
        red = (f", α↓ on {m['n_windows_alpha_reduced']}/{m['n_windows']} win "
               f"(min {m['effective_alpha_min']:.2f})") if m["n_windows_alpha_reduced"] else ""
        return f"{lvl}: max |Δ|={m['max_abs_lateral_offset_m']:.2f} m, end Δ={m['endpoint_offset_from_real_m']:.2f} m{red}"
    dev_txt = "\n".join(_dev_line(lvl) for lvl in PERTURBATION_LEVELS)
    ax_dev.set_title(f"counterfactual lateral offset from MBRA\n{dev_txt}", fontsize=9)
    ax_dev.set_xlabel("distance along MBRA path (m)")
    ax_dev.set_ylabel("lateral offset from MBRA (m,  + = left)")
    ax_dev.grid(alpha=0.25)
    ax_dev.legend(fontsize=8, framealpha=0.92)

    fig.text(0.5, 0.005, "● start   ■ end  (bird's-eye)", ha="center", fontsize=8, color="#555")
    fig.tight_layout(rect=(0, 0.02, 1, 1))

    out_dir = os.path.join(MAPS_DIR, ride_id)
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, "trajectories.csv"), index=False)
    with open(os.path.join(out_dir, "trajectory_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    fig.savefig(os.path.join(out_dir, "trajectory_map.png"), dpi=140, bbox_inches="tight")
    plt.show()
    print(f"{ride_id}: {frame_note}")
    print(f"{ride_id}: saved -> {out_dir}")
    return df, meta

In [ ]:
if not ride_ids_with_labels:
    print("No *_mbra_labels.csv under", OUTPUTS_DIR, "-- run notebooks 1 and 2 first.")

results = {}
for ride_id in ride_ids_with_labels:
    try:
        results[ride_id] = process_ride(ride_id)
    except (AssertionError, FileNotFoundError, KeyError) as e:
        print(f"{ride_id}: skipped -- {e}")